In [2]:
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [kagglehub]/4 [kagglehub]


In [6]:
import kagglehub
kagglehub.login()

In [14]:
kagglehub.dataset_download('etiennekagaba/rwandan-law', output_dir="/kaggle/input/datasets/etiennekagaba/rwandan-law")

100%|██████████| 1.35M/1.35M [00:00<00:00, 11.8MB/s]

Extracting files...


'/kaggle/input/datasets/etiennekagaba/rwandan-law'

In [9]:
!pip install -q bitsandbytes transformers peft accelerate scipy einops evaluate trl datasets

In [10]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    GenerationConfig,
    DataCollatorForLanguageModeling
)
from tqdm import tqdm
from trl import SFTTrainer
import torch
import time
import pandas as pd
import numpy as np
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import interpreter_login

interpreter_login()


    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: 00Q0-3543

    Waiting for authorization....


In [11]:
import os
# disable Weights and Biases
os.environ['WANDB_DISABLED']="true"

# Load dataset

In [15]:
TRAIN_PATH = "/kaggle/input/datasets/etiennekagaba/rwandan-law/qlora_train.jsonl"
VAL_PATH = "/kaggle/input/datasets/etiennekagaba/rwandan-law/qlora_val.jsonl"

In [16]:
df_train = pd.read_json(TRAIN_PATH, lines=True)
df_val = pd.read_json(VAL_PATH, lines=True)
hf_train = Dataset.from_pandas(df_train)
hf_val = Dataset.from_pandas(df_val)

In [17]:
df_train.head()

,instruction,context,question,response,_refusal_category
0,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 90 of Law N° 6...,I belong to an employers’ association. Does th...,"According to Article 3 of Law N° 66/2018, empl...",NaN
1,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 334 of Law N° ...,My adoptive parent filed a claim to revoke my ...,"According to Article 334 of Law N° 71/2024, th...",NaN
2,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 191 of Law N° ...,My spouse and I are under a community of prope...,"According to Article 192 of Law N° 71/2024, in...",NaN
3,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 371 of Law N° ...,I am a foreigner residing in Rwanda and want t...,"According to Article 371 of Law N° 71/2024, th...",NaN
4,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 289 of Law N° ...,I want to recognise a child who was conceived ...,"According to Article 289 of Law N° 71/2024, a ...",NaN


In [18]:
df_val.head()

,instruction,context,question,response,_refusal_category
0,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 79 of Law N° 6...,My penalty decision became final while I was a...,"According to Article 79 of Law N° 68/2018, whe...",NaN
1,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 123 of Law N° ...,"I ended my own pregnancy. If I am convicted, w...","According to Article 123 of Law N° 68/2018, a ...",NaN
2,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 335 of Law N° ...,My adoption-revocation judgment has become fin...,"According to Article 337 of Law N° 71/2024, th...",NaN
3,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 148 of Law N° ...,My spouse has been harassing me because I deci...,"According to Article 148 of Law N° 68/2018, a ...",NaN
4,You are a legal assistant for Rwandan citizens...,Document [1] (Citation: Article 77 of Law N° 6...,"I was ordered to pay a fine of FRW 500,000. Af...","According to Article 77 of Law N° 68/2018, a f...",NaN


# Create Bitsandbytes configuration

In [19]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

# Load the Pre-Trained model

In [20]:
model_name='Qwen/Qwen2.5-0.5B-Instruct'
device_map = {"": 0}
original_model = AutoModelForCausalLM.from_pretrained(model_name, 
                                                      device_map=device_map,
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Tokenization

In [21]:
tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True,padding_side="left",add_eos_token=True,add_bos_token=True,use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# Test the Model with Zero Shot Inferencing

In [22]:
def gen(model, prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id
    )
    
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [23]:
%%time
from transformers import set_seed
seed = 42
set_seed(seed)

index = 3 

instruction = df_val.iloc[index]['instruction']
context = df_val.iloc[index]['context']
question = df_val.iloc[index]['question']
baseline_response = df_val.iloc[index]['response']

formatted_prompt = f"{instruction}\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n"

res = gen(original_model, formatted_prompt, max_new_tokens=150)

output = res[0].split('Answer:\n')[-1].strip()

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{formatted_prompt}')
print(dash_line)
print(f'BASELINE HUMAN RESPONSE:\n{baseline_response}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:
You are a legal assistant for Rwandan citizens. Answer the user's question clearly and naturally using ONLY the provided legal text. You must explicitly cite the relevant Article numbers in your response. If the provided text does not contain the answer, you must state: 'I cannot answer this question based on the provided Rwandan legal context.'

Context:
Document [1] (Citation: Article 148 of Law N° 68/2018):
Article 148: Denial of freedom to practice family planning
Any person who, on grounds of marriage, commits violence or harassment against his/her spouse because of the spouse’s decision to practice family planning, commits an offence. Upon conviction, he/she is liable to imprisonment for a term of not less than two (2) months and not more than six (6) months.

Document [2] (Citation: Article 147 of Law N° 68/2018):
Article 147: Harassment of a spouse
Any person who ha

# Pre-process the dataset

In [24]:
from functools import partial

In [25]:
def format_prompt(row, with_context=True, include_response=True):
    """Formats a single row using the tokenizer's chat template."""
    instruction = row["instruction"]
    question = row["question"]
    context = row.get("context", "")

    if with_context and isinstance(context, str) and context.strip():
        user_content = f"Context:\n{context.strip()}\n\nQuestion:\n{question}"
    else:
        user_content = f"Context:\n\nQuestion:\n{question}"

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]

    if include_response and "response" in row and pd.notna(row["response"]):
        messages.append({"role": "assistant", "content": str(row["response"])})

    row['text'] = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=not include_response
        )
    return row

In [26]:
def analyze_token_lengths(df: pd.DataFrame, split_name: str):
    """Computes and displays token length statistics for a given dataframe."""
    print(f"\n{'='*20} {split_name.upper()} DATASET {'='*20}")

    records = df.to_dict(orient="records")

    train_texts = [
        format_prompt(row.copy(), with_context=True, include_response=True)["text"] 
        for row in records
    ]
    train_lengths = [len(tokens) for tokens in tokenizer(train_texts)["input_ids"]]

    eval_rag_texts = [
        format_prompt(row.copy(), with_context=True, include_response=False)["text"] 
        for row in records
    ]
    eval_rag_lengths = [len(tokens) for tokens in tokenizer(eval_rag_texts)["input_ids"]]

    eval_norag_texts = [
        format_prompt(row.copy(), with_context=False, include_response=False)["text"] 
        for row in records
    ]
    eval_norag_lengths = [len(tokens) for tokens in tokenizer(eval_norag_texts)["input_ids"]]

    s_train = pd.Series(train_lengths)
    s_eval_rag = pd.Series(eval_rag_lengths)
    s_eval_norag = pd.Series(eval_norag_lengths)

    stats = pd.DataFrame(
        {
            "Training (Full Sequence)": [
                s_train.min(),
                s_train.median(),
                s_train.quantile(0.95),
                s_train.quantile(0.99),
                s_train.max(),
            ],
            "Eval (With RAG Prompt)": [
                s_eval_rag.min(),
                s_eval_rag.median(),
                s_eval_rag.quantile(0.95),
                s_eval_rag.quantile(0.99),
                s_eval_rag.max(),
            ],
            "Eval (No-RAG Prompt)": [
                s_eval_norag.min(),
                s_eval_norag.median(),
                s_eval_norag.quantile(0.95),
                s_eval_norag.quantile(0.99),
                s_eval_norag.max(),
            ],
        },
        index=["Min", "Median", "P95", "P99", "Max"],
    )

    print(stats.round(1))
    return int(s_train.max())

In [27]:
# Run analysis
max_train_tokens = analyze_token_lengths(df_train, "Train")
max_val_tokens = analyze_token_lengths(df_val, "Validation")

global_max = max(max_train_tokens, max_val_tokens)
print(f"\nAbsolute Maximum Sequence Length Across Splits: {global_max} tokens")


==================== TRAIN DATASET ====================
        Training (Full Sequence)  Eval (With RAG Prompt)  Eval (No-RAG Prompt)
Min                        332.0                   303.0                  95.0
Median                     734.0                   653.0                 115.0
P95                       1206.9                  1108.9                 131.0
P99                       1555.2                  1433.2                 138.0
Max                       1820.0                  1705.0                 163.0

==================== VALIDATION DATASET ====================
        Training (Full Sequence)  Eval (With RAG Prompt)  Eval (No-RAG Prompt)
Min                        378.0                   316.0                  99.0
Median                     722.0                   639.0                 115.0
P95                       1147.9                  1047.7                 133.3
P99                       1508.0                  1385.5                 142.1
Max         

In [28]:
def prepare_dataset_for_trainer(dataset, tokenizer, max_length=2048):
    """
    Used for BOTH train_dataset and eval_dataset inside transformers.Trainer.
    Always includes context and response, tokenizes, and removes strings.
    Maintains the existing shuffled order of the input dataset.
    """
    format_func = partial(format_prompt, with_context=True, include_response=True)
    formatted = dataset.map(format_func)

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"], 
            max_length=max_length, 
            truncation=True, 
            padding=False
        )

    return formatted.map(
        tokenize_batch, 
        batched=True, 
        remove_columns=formatted.column_names
    )

In [29]:
train_dataset = prepare_dataset_for_trainer(hf_train, tokenizer)
eval_dataset = prepare_dataset_for_trainer(hf_val, tokenizer)

Map:   0%|          | 0/1782 [00:00<?, ? examples/s]

Map:   0%|          | 0/1782 [00:00<?, ? examples/s]

Map:   0%|          | 0/199 [00:00<?, ? examples/s]

Map:   0%|          | 0/199 [00:00<?, ? examples/s]

# Preparing the model for QLoRA

In [30]:
original_model = prepare_model_for_kbit_training(original_model)

In [31]:
print(original_model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e

In [32]:
config = LoraConfig(
    r=32, #Rank
    lora_alpha=32,
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

original_model.gradient_checkpointing_enable()

peft_model = get_peft_model(original_model, config)

In [33]:
print(peft_model.print_trainable_parameters())

trainable params: 17,596,416 || all params: 511,629,184 || trainable%: 3.4393
None


# Train PEFT Adapter

In [34]:
output_dir = f'./rwandan-legal-assistant-qlora-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    warmup_steps=50,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    do_eval=True,
    gradient_checkpointing=True,
    report_to="none",
    save_total_limit=3,
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_private_repo=True,
    hub_model_id="kagaba-etienne/rwandan-legal-assistant-qlora",
    hub_strategy="end",
)

peft_model.config.use_cache = False

peft_trainer = Trainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=peft_training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
peft_trainer.train()

Step,Training Loss,Validation Loss
100,1.078200,1.036656
200,0.836482,0.802561
300,0.619690,0.599744
400,0.460306,0.462688
500,0.333974,0.377665
600,0.267766,0.315965
700,0.239742,0.273306
800,0.217935,0.254138
900,0.190893,0.233889
1000,0.151236,0.230023
